In [ ]:
%load_ext autoreload
%autoreload 2

### Eval Losses

plot training and validation losses from training

In [ ]:
import matplotlib.pyplot as plt
import json, os

In [ ]:
def load_loss(filepath):
    with open(filepath, 'r') as f:
        return json.load(f)

holders = [None, None]
for ver in [6, 7]:
    file_path = f"/scratch/users/samutiti/U54/SubCellNuc/training_V0{ver}/train_val_loss.json"
    holders[ver - 6] = load_loss(file_path)

In [ ]:
# new gen of training doesn't store loss in the same way (older version is below)
i = 1
train_holder = holders[i][list(holders[i].keys())[0]]['train']
val_holder = holders[i][list(holders[i].keys())[0]]['train']
# print(holders[i])
# print(holders[i][list(holders[i].keys())[0]]['train'])
train_loss = []
clip_loss = []
loc_loss = []
id_loss = []
for train, val in zip(train_holder, val_holder):
    train_loss.append(train['total'])
    clip_loss.append(train['clip'])
    loc_loss.append(train['loc'])
    id_loss.append(train['id'])

print(train_loss)

In [ ]:
ver = [6, 7][i]
epochs = range(1, len(train_loss) + 1)
plt.title(f'Version {ver}: Training')
plt.plot(epochs, train_loss, label='Training Loss', marker='o')
plt.plot(epochs, clip_loss, label='Clip Loss', marker='x')
plt.plot(epochs, loc_loss, label='Localization Loss')
plt.plot(epochs, id_loss, label='ID Loss')
plt.legend()
plt.show()

In [ ]:
epochs = range(1, len(holders[0][list(holders[0].keys())[0]]['train']) + 1)
fig, axs = plt.subplots(ncols=2, figsize=(10, 8))

def set_ax(ax, version, loss):
    epochs = range(1, len(loss[list(loss.keys())[0]]['train']) + 1)
    ax.set_title(f'Version {version}')
    ax.plot(epochs, loss[list(loss.keys())[0]]['train'], label='Training loss', marker='o')
    # Plot validation loss
    ax.plot(epochs, loss[list(loss.keys())[0]]['val'], label='Validation loss', marker='x')

set_ax(axs[0], 1, holders[0])
set_ax(axs[1], 2, holders[1])

plt.show()

In [ ]:
loss_range = []
for holder in holders:
    tloss = holder[list(holder.keys())[0]]['train']
    diff = max(tloss) - min(tloss)
    print(f"{max(tloss)} - {min(tloss)} --> {diff}")
    loss_range.append(diff)

#### Misc.

In [ ]:
## quickly check the storage mode of subcell embeddings
import torch 

data = torch.load('/scratch/users/samutiti/U54/embeddings/all_harmonized_features_microscope_vit.pth', weights_only=False)

In [ ]:
print(type(data))
print(type(data[0]))
print(type(data[1]))

In [ ]:
print(data[1].shape)

### Eval inference outputs

evaluate embeddings / set up for PCA and Umapping

In [14]:
import anndata as ad 

training_v = 5
adata = ad.read_h5ad(f"/scratch/users/samutiti/U54/SubCellNuc/training_V0{training_v}/inference_U2OS.h5ad")
adata = adata[adata.obs['gene_name'] != '']

In [15]:
print(adata)
print(adata.obs['gene_name'])
# print(adata.obs['gene_name'])

View of AnnData object with n_obs × n_vars = 285974 × 1280
    obs: 'gene_name', 'gene_idx', 'locations', 'atlas_name'
    obsm: 'h'
0         GPR173
1         GPR173
2         GPR173
3         GPR173
4         GPR173
           ...  
293101     PRKCE
293102     PRKCE
293103     PRKCE
293104     PRKCE
293105     PRKCE
Name: gene_name, Length: 285974, dtype: category
Categories (10971, object): ['A1CF', 'A4GALT', 'AAAS', 'AADAT', ..., 'ZYG11B', 'ZYX', 'ZZEF1', 'ZZZ3']


#### Analysis

In [ ]:
import scanpy as sc

NUM_PCS = 50
LEI_RES = 0.3

sc.pp.pca(
        adata,
        n_comps=NUM_PCS,
        svd_solver="arpack"
    )
print('pca complete')

sc.pp.neighbors(
        adata,
        n_neighbors=30,
        n_pcs=NUM_PCS,
    )
print('nieghbors complete')

sc.tl.leiden(
        adata,
        resolution=LEI_RES,
        key_added=f'leiden_{LEI_RES}'
    )
print('leiden complete')

sc.tl.umap(
        adata,
        min_dist=0.1,
    )

print('umap complete')

adata.obs["umap_x"] = adata.obsm["X_umap"][:, 0]
adata.obs["umap_y"] = adata.obsm["X_umap"][:, 1]

### Visualize Umap
sc.pl.umap(
        adata,
        color=[f"leiden_{LEI_RES}"],
        show=False,
        save=f"_training_V0{training_v}_mlp_embed.png"
    )


### Generate UMAPs

In [16]:
import anndata as ad
import seaborn as sns
import numpy as np

In [17]:
### Color By XAP
# Normalize function
def normalize_gene(g):
    return g.replace("-", "").replace("_", "").strip().upper()

# ADD TO THIS LIST
xap_set_normalized = {normalize_gene(g) for g in {
"HMGB1", "ILF2", "XRN2", "POLDIP3", "RNF20", "FUBP3", "HNRPLL", "ILF3",
"SSB", "SRRT", "EIF4A3", "RNMT", "PTBP2", "SAP18", "WTAP", "KHDRBS1",
"ERH", "CIZ1", "HNRNPH3", "SYNCRIP", "NONO", "FUS", "DHX9", "RALY",
"DDX17", "HNRPDL", "KHSRP", "RNF2", "HNRNPA3", "HNRNPUL2", "HNRNPD",
"DDX5", "DDX39B", "PTBP1", "HNRNPR", "ZFR", "ELAVL1", "SRSF2",
"HNRNPM", "RBMXL1", "TARDBP", "MYEF2", "HNRNPU", "HNRNPC",
"HNRNPL", "HNRNPAB", "HNRNPK", "HNRNPA2B1", "HNRNPA1",
"HNRNPA0", "RBM14", "TRA2B", "SFPQ", "IGF2BP1", "SARNP",
"SRSF3", "MATR3", "SRSF5", "SPEN", "RBM15", "RBM3",
"SRSF7", "RBFOX2", "SRSF9", "SAFB", "THOC4", "DDX39A",
"CELF1", "PCGF5", "RYBP", "TRIM71", "SRSF10", "RBM4",
"YTHDC1", "TRIM6", "LIN28A", "SLTM", "SAFB2", "L1TD1",
"MYBBP1A", "IGF2BP3", "SPEN", "RNF20", "RNF20", "MATR3", "HNRNPA0"
}}

def is_xap(gene_string):
    # genes = [normalize_gene(g) for g in gene_string.split(",")]
    gene = normalize_gene(gene_string)
    if gene in xap_set_normalized:
        return "XAPs"
    return "Other Proteins"

adata.obs['XAP'] = [is_xap(g) for g in adata.obs['gene_name']]
print(np.unique(adata.obs['XAP']))

/tmp/ipykernel_121717/739343265.py:30: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs['XAP'] = [is_xap(g) for g in adata.obs['gene_name']]


['A1CF' 'A4GALT' 'AAAS' ... 'ZYX' 'ZZEF1' 'ZZZ3']
['Other Proteins' 'XAPs']


In [ ]:
train_v = 5
adata_path = f"/scratch/users/samutiti/U54/SubCellNuc/training_V0{train_v}/inference_U2OS_analyzed.h5ad"
adata = ad.read_h5ad(adata_path)
print(adata)

In [ ]:
sns.scatterplot(
    data=adata.obs,
    x="umap_x",
    y="umap_y",
    hue="leiden_0.3",
    s=0.5,
    alpha=0.8
)